## Content-Based Recommender B (with metadata and review text)

This notebook implements a content-based filtering recommender system that uses both metadata and review text. Each restaurant is represented by a feature vector that combines:

- Multi-hot encoded categories
- One-hot encoded price range (RestaurantsPriceRange2)
- Binary attributes (GoodForKids, RestaurantsAttire)
- Transformer-based review embeddings

User profiles are built by averaging the feature vectors of restaurants each user has interacted with in the training set. Recommendations are generated by computing cosine similarity between a user's profile vector and all candidate restaurant vectors (the restaurants not seen in training).

### Loading Data

Loads the three CSV files:

- restaurants_santa_barbara.csv
- test_reviews_santa_barbara.csv
- train_reviews_santa_barbara.csv

In [21]:
import pandas as pd
import numpy as np

#resturants
df_rests = pd.read_csv(
    "restaurants_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_rests.columns = df_rests.iloc[0]
df_rests = df_rests[1:]
df_rests = df_rests.reset_index(drop=True)

#test
df_test = pd.read_csv(
    "test_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_test.columns = df_test.iloc[0]
df_test = df_test[1:]
df_test = df_test.reset_index(drop=True)

#train
df_train = pd.read_csv(
    "train_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_train.columns = df_train.iloc[0]
df_train = df_train[1:]
df_train = df_train.reset_index(drop=True)


### Encode restaurant categories (multi-hot encodings)

Each restaurant has a "categories" field with a list of tags

Use multi-hot encoding: each unique category across all restaurants becomes one binary column. A restaurant gets a 1 in each column corresponding to its categories and 0 everywhere else.

This is better than a transformer embedding for categories because:
- Categories are discrete labels, not semantic text
- A transformer would produce 384 dense dimensions, drowning out the smaller attribute features
- Keeps each category as an interpretable, equally-weighted binary signal


In [22]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
# Multi-hot encode restaurant categories using get_dummies

# Split comma-separated categories string into a list per restaurant
df_rests["category_list"] = df_rests["categories"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",")]
)

# get_dummies for binary encoding
category_dummies = pd.get_dummies(
    df_rests["category_list"].explode()
).groupby(level=0).max()
category_dummies = category_dummies.reindex(df_rests.index, fill_value=0)

### Encode attribute features

Chose three attributes from the restaurant metadata:

- GoodForKids (binary): users with families will prefer family-friendly restaurants
- RestaurantsAttire (binary): users who prefer comfort seek casual restaurants
- RestaurantsPriceRange2 (multi-class: 1–4): price is an important determining factor for many customers

Encoding strategy:
- Binary attributes → 0 or 1 directly
- Multi-class price range → one-hot encoding with get_dummies, so each price level gets its own column

In [23]:
#on top of the categorical embeddings, I want to add to each embedding a few values connoting attribute values

#first clean values for the attribute column
import ast
from sklearn.preprocessing import normalize
from scipy.sparse import hstack, csr_matrix

def str_to_dict(s):
    """
    Safely parse a string representation of a Python dictionary.
    Args:
        s (str): A string that may contain a dictionary literal.
    Returns:
        dict: The parsed dictionary, or an empty dict if parsing fails.
    """
    try:
        return ast.literal_eval(s)
    except: #need this or else it won't run
        return {}
df_rests['attributes_dict'] = df_rests['attributes'].apply(str_to_dict)

def clean_values(d):
    """
    Strip extra quote characters from string values in a dictionary.
    Args:
        d (dict): A dictionary whose values may contain leading/trailing quotes.
    Returns:
        dict: The same dictionary with string values without quote characters.
    """
    return {k: (v.strip("'\"") if isinstance(v, str) else v) for k, v in d.items()}
df_rests['attributes_dict'] = df_rests['attributes_dict'].apply(clean_values)

# use RestaurantsPriceRange2, GoodForKids, and RestaurantsAttire; these are all good indicators because people feel strongly about prices, about their families, and about "comfortability" of a restaurant. A person who wants to eat cheap will really appreciate a cheap recommendation, a person who has kids will really appreciate a restaurant that caters to them, and a person who likes to be comfortable/homey will really appreciate a restaurant recommendation that feels like home.

# Binary attribute: GoodForKids
df_rests["good_for_kids"] = df_rests["attributes_dict"].apply(
    lambda d: 1 if d.get("GoodForKids") == "True" else 0
)

# Binary attribute: RestaurantsAttire (casual vs not)
df_rests["casual"] = df_rests["attributes_dict"].apply(
    lambda d: 1 if d.get("RestaurantsAttire") == "casual" else 0
)

# Multi-class attribute: PriceRange — one-hot encode with get_dummies
# Restaurants with no price data are assigned "unknown" category
df_rests["price_range"] = df_rests["attributes_dict"].apply(
    lambda d: d.get("RestaurantsPriceRange2", "unknown")
)
price_dummies = pd.get_dummies(df_rests["price_range"], prefix="price")
price_dummies = price_dummies.reindex(df_rests.index, fill_value=0)

### Make weighted metadata vector

Three feature blocks (categories, price range, binary attributes) are combined into a single metadata vector with weighted concatenation. Categories are 0.6, price range are 0.25, and binary attributes are 0.15.

This ensures that the largest block (the categories) won't dominate the cosine similarity calculation just because of its larger dimensionality.

Combined vector is L2-normalized prior to computing cosine similarity.


In [24]:
X_cat = csr_matrix(category_dummies.values.astype(float))
X_attr = csr_matrix(price_dummies.values.astype(float))
X_binary = csr_matrix(df_rests[["good_for_kids", "casual"]].values.astype(float))

# set weights
weight_cat = 0.6
weight_attr = 0.25
weight_binary = 0.15

# Concatenate weighted blocks into a single matrix
X_meta_sparse = hstack([X_cat * weight_cat, X_attr * weight_attr, X_binary * weight_binary])
# L2 normalize each row so all metadata vectors have unit length
X_meta = normalize(X_meta_sparse, norm="l2", axis=1).toarray()

df_rests["meta_embedding"] = list(X_meta)
print("Metadata vector size:", X_meta.shape[1])

Metadata vector size: 232


### Aggregate reviews and embed

All training reviews for each restaurant are concatenated, then embedded using the all-MiniLM-L6-v2 sentence transformer

Aggregate reviews before embedding to produce one vector per restaurant that captures the overall language of its reviews. Restaurants with no training reviews will receive a zero vector.

Only training reviews are used. Test reviews are never seen during modeling.

In [25]:
from sentence_transformers import SentenceTransformer

embedder_model = SentenceTransformer("all-MiniLM-L6-v2")

# Aggregate all review texts per restaurant from training data
review_text_by_restaurant = df_train.groupby("business_id")["text"].apply(
    lambda texts: " ".join(texts.dropna())
).reset_index()
review_text_by_restaurant.columns = ["business_id", "aggregated_text"]

# Embed the aggregated review texts
review_embeddings = embedder_model.encode(review_text_by_restaurant["aggregated_text"].tolist())
review_text_by_restaurant["review_embedding"] = list(review_embeddings)

# Merge into restaurant dataframe
df_rests_b = df_rests.merge(
    review_text_by_restaurant[["business_id", "review_embedding"]],
    on="business_id",
    how="left"
)

# Fill restaurants with no reviews with a zero vector (same size as the model output)
embedding_size = review_embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
df_rests_b["review_embedding"] = df_rests_b["review_embedding"].apply(
    lambda x: x if isinstance(x, np.ndarray) else np.zeros(embedding_size)
)

/Users/troyhealey/PycharmProjects/RecomSystemsClass/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Make final feature vector

Create the final restaurant vector by horizontally concatenating the normalized metadata vector (categories, price range, binary attributes) and the review embedding.

The concatenated vector is L2 normalized again so the full vector has unit length, ensuring correct calculation of cosine similarity.

Final vector composition:
- Metadata block: weighted and normalized (size = num_categories + num_price_levels + 2)
- Review embedding block: 384 dimensions from all-MiniLM-L6-v2
- Total: metadata_size + 384


In [26]:
# pull meta and review matrices
X_meta = np.vstack(df_rests_b["meta_embedding"].values)
review_matrix = np.vstack(df_rests_b["review_embedding"].values)

# concatenate and L2 normalize final vector
X_final = np.hstack([X_meta, review_matrix])
X_final = normalize(X_final, norm="l2", axis=1)

df_rests_b["new_embedding"] = list(X_final)
print("Final vector size (B):", X_final.shape[1])
# Should be: metadata_size + 384

Final vector size (B): 616


### Build user profiles

Each user is represented by a profile vector computed as the mean of the feature vectors of restaurants they interacted with in the training set.


In [27]:
df_train['stars'] = pd.to_numeric(df_train['stars'])

# Compute per-user mean rating and adjusted rating
user_means = df_train.groupby("user_id")["stars"].mean()
df_train = df_train.merge(user_means.rename("user_mean"), on="user_id")
df_train["adjusted_rating"] = df_train["stars"] - df_train["user_mean"]

# Join training interactions with restaurant embeddings
merge = df_train.merge(df_rests_b, on="business_id")
merge = merge.set_index("business_id")

# Build user profiles as the mean embedding of all restaurants the user interacted with
user_profiles = {}
for user, group in df_train.groupby("user_id"):
    vectors = merge.loc[group["business_id"], "new_embedding"]
    user_profiles[user] = np.vstack(vectors).mean(axis=0)


### Recommendation function

Given a user, compute cosine similarity between their profile vector and all candidate restaurant vectors, then return the top-k most similar restaurants.

Candidate restaurants are all restaurants that have not been seen yet by the user

In [28]:
#get recommendations
from sklearn.metrics.pairwise import cosine_similarity

def recommend_restaurants(user_id, user_embeddings, restaurants_df, k, seen_business_ids=None):
    """
    Generate top-k restaurant recommendations for a given user using cosine similarity.

    Args:
        user_id (str): The ID of the user to generate recommendations for.
        user_embeddings (dict): Mapping from user_id to their profile vector (np.ndarray).
        restaurants_df (pd.DataFrame): DataFrame containing restaurant feature vectors
            in the 'new_embedding' column.
        k (int): Number of top recommendations to return.
        seen_business_ids (set, optional): Set of business IDs the user has already interacted
            with in training. These will be excluded from recommendations. Defaults to None.

    Returns:
        pd.DataFrame: A DataFrame with columns ['business_id', 'name', 'similarity'],
            sorted by descending similarity score.
    """
    if seen_business_ids:
        restaurants_df = restaurants_df[~restaurants_df["business_id"].isin(seen_business_ids)].reset_index(drop=True)

    user_vec = user_embeddings[user_id].reshape(1, -1)
    restaurant_matrix = np.vstack(restaurants_df['new_embedding'].values)

    similarities = cosine_similarity(user_vec, restaurant_matrix)[0]
    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = restaurants_df[['business_id', 'name']].iloc[top_k_idx].copy()
    results['similarity'] = similarities[top_k_idx]

    return results

### Evaluation Metrics: Hit@k and NDCG@k

**Hit@K**: Measures whether the true test restaurant appears in the top-K recommendations

**NDCG@K**: Measures the ranking quality by assigning higher scores when the true restaurant is ranked higher in the recommendation list

In [29]:
#now we can do accuracy metrics. first is NDCG@k
def ndcg_at_k(recommended_ids, true_id, k):
    """
    Compute the NDCG score for a single user at cutoff k.
    Since each user has exactly one held-out item, NDCG simplifies to
    1 / log2(rank + 1) if the item appears in the top-k, else 0.

    Args:
        recommended_ids (list): Ordered list of recommended business IDs.
        true_id (str): The held-out business ID for this user.
        k (int): Cutoff rank.

    Returns:
        float: NDCG score in [0, 1].
    """
    try:
        rank = recommended_ids.index(true_id) + 1
    except ValueError:
        return 0.0

    return 1 / np.log2(rank + 1)

def evaluate_ndcg(user_embeddings, restaurants_df, test_df, k):
    """
    Compute mean NDCG@k across all users in the test set.

    Args:
        user_embeddings (dict): Mapping from user_id to profile vector.
        restaurants_df (pd.DataFrame): Restaurant feature vectors.
        test_df (pd.DataFrame): Test interactions with columns ['user_id', 'business_id'].
        k (int): Cutoff rank.

    Returns:
        float: Mean NDCG@k across all test users.
    """
    scores = []

    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']

        seen = set(df_train[df_train["user_id"] == row["user_id"]]["business_id"])
        recs = recommend_restaurants(user_id, user_embeddings, restaurants_df, k, seen_business_ids=seen)
        recommended_ids = recs['business_id'].tolist()
        score = ndcg_at_k(recommended_ids, true_business_id, k)
        scores.append(score)

    return np.mean(scores)

#then hit@k
def hit_at_k(recommended_ids, true_id):
    """
    Compute the Hit score for a single user: 1 if the held-out item is in the list, else 0.

    Args:
        recommended_ids (list): Ordered list of recommended business IDs.
        true_id (str): The held-out business ID for this user.

    Returns:
        int: 1 if true_id is in recommended_ids, 0 otherwise.
    """
    return int(true_id in recommended_ids)

def evaluate_hit_at_k(user_embeddings, restaurants_df, test_df, k):
    """
    Compute mean Hit@k across all users in the test set.

    Args:
        user_embeddings (dict): Mapping from user_id to profile vector.
        restaurants_df (pd.DataFrame): Restaurant feature vectors.
        test_df (pd.DataFrame): Test interactions with columns ['user_id', 'business_id'].
        k (int): Cutoff rank.

    Returns:
        float: Mean Hit@k across all test users.
    """
    hits = []
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']

        seen = set(df_train[df_train["user_id"] == row["user_id"]]["business_id"])
        recs = recommend_restaurants(user_id, user_embeddings, restaurants_df, k, seen_business_ids=seen)
        recommended_ids = recs['business_id'].tolist()

        hit = hit_at_k(recommended_ids, true_business_id)
        hits.append(hit)

    return np.mean(hits)

### Printing Evaluation Metrics

Evaluating Hit@10, Hit@20, Hit@30 and NDCG@10, NDCG@20, NDCG@30


In [30]:
print("Content-Based Recommender B (Metadata and Review Text): Evaluation Metrics")
for k in [10, 20, 30]:
    print(f"Hit@{k}:", evaluate_hit_at_k(user_profiles, df_rests_b, df_test, k), f"\t\tNDCG@{k}:", evaluate_ndcg(user_profiles, df_rests_b, df_test, k))

Content-Based Recommender B (Metadata and Review Text): Evaluation Metrics
Hit@10: 0.02936888148302437 		NDCG@10: 0.01460653258592255
Hit@20: 0.05311393459695897 		NDCG@20: 0.020546715064386455
Hit@30: 0.08019162674442824 		NDCG@30: 0.026278475535812106
